In [ ]:
from pathlib import Path
from typing import Literal
import string
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from common import load_data, ggcms, ssps, load_spam_w

In [ ]:
spam = load_spam_w(Path("../aux_data/spam2020_V2r2_global_H_MAIZ_A.tif"))

In [ ]:
features_plot = ["TMXav", "PRCPsum", "RADsum", "KDD", "CMDgt0", "PETsum"]
ggcms_plot = [x for x in ggcms if x != "GSWP3"]
ssps_plot = ["ssp245", "ssp370"]

In [ ]:
def apply_area_weights(data: pl.DataFrame) -> pl.DataFrame:
    return (
        data
        .join(spam, on=["LAT", "LON"])
        .with_columns([
            (pl.col(x) / pl.col("LEN")).alias(x)
            for x in features_plot
            if x != "TMXav"
        ])
        .group_by("YR")
        .agg([
            (
                (pl.col(x) * pl.col("W")).sum()
                / pl.col("W").sum()
            ).alias(x)
            for x in features_plot
        ])
        .sort("YR")
    )

def format_feature_name(name: str) -> str:
    return {
        "TMXav": "Δ Max. temperature [°C/day]", 
        "PRCPsum": "Δ Precipitation [mm/day]", 
        "RADsum": "Δ Radiation [MJ/m²/day]", 
        "KDD": "Δ Killing degree days [% of GS]", 
        "PETsum": "Δ PET [mm/day]",
        "CMDgt0": "Δ Days where CMD > 0 [% of GS]"
    }[name]

In [ ]:
# Create area-weighted baseline
hist_from = 1985
hist_to = 2014

baselines = {}
for ggcm in ggcms_plot:

    hist = apply_area_weights(
        load_data("historical", ggcm, "mai")
        .filter(
            pl.col("YR").is_between(hist_from, hist_to),
            pl.col("PERIOD") == "gs",
        )
    )
    
    baselines[ggcm] = hist


In [ ]:
deltas = {}
for ssp in ssps_plot:
    deltas[ssp] = {}
    for gcm in ggcms_plot:
        future = apply_area_weights(
            load_data(ssp, gcm, "mai")
            .filter(pl.col("PERIOD") == "gs")
        )
        baseline = baselines[gcm].mean()
        delta_hist = (
            baselines[gcm]
            .with_columns([
                (pl.col(x) - pl.lit(baseline[x])).alias(x)
                for x in features_plot
            ])
            .with_columns([
                (pl.col(x) * 100).alias(x) 
                for x in ["KDD", "CMDgt0"]
            ])
        )
        delta_future = (
            future
            .with_columns([
                (pl.col(x) - pl.lit(baseline[x])).alias(x)
                for x in features_plot
            ])
            .with_columns([
                (pl.col(x) * 100).alias(x) 
                for x in ["KDD", "CMDgt0"]
            ])
        )
        deltas[ssp][gcm] = pl.concat([delta_hist, delta_future], how="vertical")

In [ ]:
colors = list(mcolors.TABLEAU_COLORS)
labels = string.ascii_lowercase

num_rows = len(features_plot)
num_cols = len(ssps_plot)
plt.rcParams.update({'font.size': 14})
fig, axs = plt.subplots(num_rows, num_cols, sharey="row", layout="compressed", figsize=(15, 20))

for row, feature in enumerate(features_plot):
    
    for col, ssp in enumerate(ssps_plot):
        
        ax = axs[row, col]
        for gcm in ggcms_plot:
           ax.plot(deltas[ssp][gcm]["YR"], deltas[ssp][gcm][feature], label=gcm)
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)

        if col == 0:
            ax.set_ylabel(format_feature_name(feature))
        if row == 0:
            ax.set_title(ssp.upper())

        # Panel label
        panel = row * num_cols + col
        ax.text(
            0.02,
            0.95,
            labels[panel],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=16,
            fontweight="bold",
        )

handles = [
    Patch(facecolor=color, label=label)
    for color, label in zip(colors, ggcms_plot)
]

fig.legend(
    handles=handles,
    labels=ggcms_plot,
    ncol=len(ggcms_plot),
    loc="upper center",
    bbox_to_anchor=(0.5, 1.00),
    frameon=False,
    fontsize=14,
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

fig.savefig("features_lines.png")
fig.savefig("features_lines.pdf")

fig.show()

In [ ]:
from scipy.stats import linregress

pvals = []
feat = "PRCPsum"
for ssp in ssps_plot:
    for gcm in ggcms_plot:
        data_feat = deltas[ssp][gcm]
        
        res = linregress(
            data_feat["YR"].to_numpy(),
            data_feat[feat].to_numpy(),
        )

        print(
            f"{ssp:7} {gcm:15} "
            f"slope={res.slope:.4f}, "
            f"intercept={res.intercept:.4f}, "
            f"p={res.pvalue:.3e}"
        )
        pvals.append(res.pvalue)